# Postman Generator!

In [ ]:
# imports

import os
import requests
import re
import xml.etree.ElementTree as ET
import yaml
import subprocess
import tempfile
import hashlib
import zipfile
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
from typing import List
from typing import Optional




In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

claude_model = "claude-opus-4-8"
gpt_model = "gpt-5.4"
gpt_support = "gpt-5-mini"

repository = "C:/Users/ME36352/MyFolder/REPOS/ACE"
apic_api = "C:/Users/ME36352/MyFolder/REPOS/APIC/insurance-policies-api-insurance-events/src/main/resources"
services = ["common-integration-business-event-publish", 
            "insurance-integration-business-event-enrichment",
            "insurance-integration-business-event-call-sfdc",
            "clients-prospects-marketing-integration-technical-dp-insurance-events",
            "insurance-integration-business-event-call-dpm",
            "insurance-policies-integration-bolttech",
            "common-integration-shlib-sfdcwebservices"] 
services_path = []
for service in services:
    service_path = services_path.append(repository + "/" + service)

with open("taskDescription.txt", "r", encoding="utf-8") as f:
    task_description = f.read()
with open("mainSystemPrompt.txt", "r", encoding="utf-8") as f:
    main_system_prompt = f.read()
#with open("supportSystemPrompt.txt", "r", encoding="utf-8") as f:
#   support_system_prompt = f.read()
with open("explorerSystemPrompt.txt", "r", encoding="utf-8") as f:
    explorer_system_prompt = f.read()
    
request = task_description

In [ ]:
def save_string_to_txtFile(content: str, filename: str):

    filename += ".txt"
    output_path =  Path.cwd() / filename
    output_path.write_text(content, encoding="utf-8")

In [ ]:
def read_directories_contents(base_paths: List[str], only_esql: bool = False ) -> str:
    """
    Recursively reads all files under each path in base_paths and returns
    a single string containing relative file paths followed by their contents.

    :param base_paths: List of root directories to scan
    :return: Aggregated string of file paths and contents
    """
    result_parts = []

    for base_path in base_paths:
        for root, _, files in os.walk(base_path):
            for file_name in files:
                if only_esql and file_name.endswith(".esql"):
                    full_path = os.path.join(root, file_name)
                    rel_path = os.path.relpath(full_path, base_path)

                    try:
                        with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                            content = f.read()
                    except Exception as e:
                        # Skip unreadable files but record the issue
                        content = f"[Error reading file: {e}]"

                    result_parts.append(
                        f"File directory: {rel_path}\n\n"
                        f"{content}\n"
                    )
                elif  only_esql == False:
                    full_path = os.path.join(root, file_name)
                    rel_path = os.path.relpath(full_path, base_path)

                    try:
                        with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                            content = f.read()
                    except Exception as e:
                        # Skip unreadable files but record the issue
                        content = f"[Error reading file: {e}]"

                    result_parts.append(
                        f"File directory: {rel_path}\n\n"
                        f"{content}\n"
                    )

    return "\n".join(result_parts)

In [ ]:
services_code = read_directories_contents(services_path, True)

In [ ]:
def collect_service_with_dependencies(
    repository_path: str,
    service_name: str,
    cfr_jar_path: str = "cfr.jar",
    include_java: bool = False
) -> str:
    """
    Collect all files from a service and its transitive dependencies.

    Dependency resolution:
    - Maven pom.xml artifactId index
    - Eclipse .project <projects> dependencies

    Modes:
    - include_java=False (default):
        Works like the original function:
        * normal text files are included
        * .jar files are reported as not readable
        * .class files are reported as not readable
        * other binary files are reported as not readable
    - include_java=True:
        * normal text files are included
        * .class files are decompiled with CFR and emitted as .java
        * .jar files are decompiled with CFR and only emitted as .java entries
        * duplicate Java types are emitted only once

    Java duplicate resolution priority when include_java=True:
    1. real .java source file
    2. decompiled standalone .class
    3. decompiled .jar entry
    """
    repository = Path(repository_path).resolve()
    service_root = (repository / service_name).resolve()
    cfr_jar = Path(cfr_jar_path).resolve()

    if not service_root.exists():
        raise FileNotFoundError(f"Service '{service_name}' not found in repository.")

    if include_java and not cfr_jar.exists():
        raise FileNotFoundError(f"CFR jar not found: {cfr_jar}")

    # ------------------------------------------------------------------
    # Ignore rules
    # ------------------------------------------------------------------
    GIT_FILES = {".gitignore", ".gitattributes", ".gitmodules"}
    BINARY_EXTENSIONS = {
        ".exe", ".dll", ".zip", ".war", ".ear", ".png", ".jpg", ".jpeg",
        ".gif", ".bmp", ".ico", ".pdf", ".so", ".o", ".obj", ".a", ".lib"
    }

    def should_skip(path: Path) -> bool:
        return ".git" in path.parts or path.name in GIT_FILES

    # ------------------------------------------------------------------
    # Index repository once
    # ------------------------------------------------------------------
    artifact_to_dirs = {}
    project_files = []

    for file in repository.rglob("*"):
        if should_skip(file):
            continue
        if not file.is_file():
            continue

        if file.name == "pom.xml":
            try:
                tree = ET.parse(file)
                root = tree.getroot()

                if root.tag.startswith("{"):
                    uri = root.tag.split("}")[0][1:]
                    ns = {"m": uri}
                    artifact = root.find("m:artifactId", ns)
                else:
                    artifact = root.find("artifactId")

                if artifact is not None and artifact.text:
                    artifact_to_dirs.setdefault(
                        artifact.text.strip(), []
                    ).append(file.parent.resolve())
            except Exception:
                pass

        elif file.name == ".project":
            project_files.append(file.resolve())

    # ------------------------------------------------------------------
    # Resolve .project belonging to a directory
    # ------------------------------------------------------------------
    def find_project_file(project_dir: Path) -> Optional[Path]:
        project_dir = project_dir.resolve()
        direct = project_dir / ".project"
        if direct.exists():
            return direct.resolve()

        for pf in project_files:
            try:
                pf.relative_to(project_dir)
                return pf
            except Exception:
                continue
        return None

    # ------------------------------------------------------------------
    # Dependency parsing
    # ------------------------------------------------------------------
    def extract_dependencies(project_file: Optional[Path]):
        if not project_file:
            return []

        deps = []
        try:
            tree = ET.parse(project_file)
            root = tree.getroot()
            projects = root.find("projects")
            if projects is None:
                return deps

            for p in projects.findall("project"):
                if p.text:
                    deps.append(p.text.strip())
        except Exception:
            pass

        return deps

    # ------------------------------------------------------------------
    # Java identity parsing
    # ------------------------------------------------------------------
    def extract_java_identity(java_source: str, fallback_name: str) -> str:
        package_match = re.search(
            r'^\s*package\s+([A-Za-z_][\w.]*)\s*;',
            java_source,
            flags=re.MULTILINE
        )
        type_match = re.search(
            r'\b(class|interface|enum|record)\s+([A-Za-z_]\w*)\b',
            java_source
        )

        package_name = package_match.group(1) if package_match else ""
        type_name = type_match.group(2) if type_match else fallback_name

        return f"{package_name}.{type_name}" if package_name else type_name

    # ------------------------------------------------------------------
    # Output storage
    # ------------------------------------------------------------------
    visited_projects = set()
    processed_files = set()
    output_records = []
    emitted_text_labels = set()

    # Used only when include_java=True
    java_candidates = {}
    java_order_seen = set()

    def add_text_record(label: str, content: str):
        if label in emitted_text_labels:
            return
        emitted_text_labels.add(label)
        output_records.append(("text", label, content))

    def add_java_candidate(label: str, java_source: str, fallback_name: str, priority: int):
        if not java_source or not java_source.strip():
            return

        identity = extract_java_identity(java_source, fallback_name)
        existing = java_candidates.get(identity)

        if existing is None:
            java_candidates[identity] = {
                "label": label,
                "content": java_source,
                "priority": priority
            }
            if identity not in java_order_seen:
                java_order_seen.add(identity)
                output_records.append(("java", identity, None))
            return

        if priority > existing["priority"]:
            java_candidates[identity] = {
                "label": label,
                "content": java_source,
                "priority": priority
            }

    # ------------------------------------------------------------------
    # CFR helpers
    # ------------------------------------------------------------------
    def run_cfr_on_class_file(class_file: Path) -> Optional[str]:
        try:
            result = subprocess.run(
                [
                    "java",
                    "-jar",
                    str(cfr_jar),
                    str(class_file),
                    "--silent",
                    "true"
                ],
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="ignore",
                check=False
            )
            source = (result.stdout or "").strip()
            return source if source else None
        except Exception:
            return None

    def run_cfr_on_jar_file(jar_file: Path):
        results = []

        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                outdir = Path(tmpdir) / "cfr_out"
                outdir.mkdir(parents=True, exist_ok=True)

                subprocess.run(
                    [
                        "java",
                        "-jar",
                        str(cfr_jar),
                        str(jar_file),
                        "--silent",
                        "true",
                        "--outputdir",
                        str(outdir)
                    ],
                    capture_output=True,
                    text=True,
                    encoding="utf-8",
                    errors="ignore",
                    check=False
                )

                for java_file in sorted(outdir.rglob("*.java")):
                    try:
                        java_rel = java_file.relative_to(outdir).as_posix()
                    except Exception:
                        java_rel = java_file.name

                    try:
                        java_source = java_file.read_text(
                            encoding="utf-8",
                            errors="ignore"
                        )
                    except Exception:
                        continue

                    if java_source.strip():
                        results.append((java_rel, java_source))
        except Exception:
            return []

        return results

    # ------------------------------------------------------------------
    # File reader
    # ------------------------------------------------------------------
    def read_files(base_dir: Path):
        for file in sorted(base_dir.rglob("*")):
            if should_skip(file):
                continue
            if not file.is_file():
                continue

            resolved_file = file.resolve()
            if resolved_file in processed_files:
                continue
            processed_files.add(resolved_file)

            rel = file.relative_to(repository).as_posix()
            suffix = file.suffix.lower()

            # ----------------------------------------------------------
            # include_java=True mode
            # ----------------------------------------------------------
            if include_java:
                if suffix == ".jar":
                    decompiled_files = run_cfr_on_jar_file(file)
                    for java_rel, java_source in decompiled_files:
                        label = f"{rel}!/{java_rel}"
                        add_java_candidate(
                            label=label,
                            java_source=java_source,
                            fallback_name=Path(java_rel).stem,
                            priority=1
                        )
                    continue

                if suffix == ".class":
                    java_source = run_cfr_on_class_file(file)
                    if java_source:
                        java_label = str(Path(rel).with_suffix(".java")).replace("\\", "/")
                        add_java_candidate(
                            label=java_label,
                            java_source=java_source,
                            fallback_name=file.stem,
                            priority=2
                        )
                    continue

                if suffix in BINARY_EXTENSIONS:
                    continue

                try:
                    content = file.read_text(encoding="utf-8", errors="ignore")
                except Exception:
                    continue

                if suffix == ".java":
                    add_java_candidate(
                        label=rel,
                        java_source=content,
                        fallback_name=file.stem,
                        priority=3
                    )
                else:
                    add_text_record(rel, content)

            # ----------------------------------------------------------
            # include_java=False mode -> original behavior
            # ----------------------------------------------------------
            else:
                if suffix == ".jar":
                    add_text_record(rel, "--- JAR CONTENT NOT READABLE ---")
                    continue

                if suffix == ".class":
                    add_text_record(rel, "--- JAVA CLASS NOT READABLE ---")
                    continue

                if suffix in BINARY_EXTENSIONS:
                    add_text_record(rel, "--- BINARY FILE NOT READABLE ---")
                    continue

                try:
                    content = file.read_text(encoding="utf-8", errors="ignore")
                except Exception:
                    continue

                add_text_record(rel, content)

    # ------------------------------------------------------------------
    # Project resolver
    # ------------------------------------------------------------------
    def resolve_project(project_dir: Path):
        project_dir = project_dir.resolve()
        if project_dir in visited_projects:
            return

        visited_projects.add(project_dir)

        read_files(project_dir)

        project_file = find_project_file(project_dir)
        deps = extract_dependencies(project_file)

        for dep in deps:
            candidate_dirs = artifact_to_dirs.get(dep, [])
            for dep_dir in candidate_dirs:
                resolve_project(dep_dir)

    # ------------------------------------------------------------------
    # Run + render
    # ------------------------------------------------------------------
    resolve_project(service_root)

    rendered = []
    for kind, key, value in output_records:
        if kind == "text":
            rendered.append(
                f"File directory: {key}\n\n"
                f"{value}\n\n"
                + "=" * 80 + "\n\n"
            )
        elif kind == "java" and include_java:
            candidate = java_candidates.get(key)
            if not candidate:
                continue
            rendered.append(
                f"File directory: {candidate['label']}\n\n"
                f"{candidate['content']}\n\n"
                + "=" * 80 + "\n\n"
            )

    return "".join(rendered)

In [ ]:
services_code_with_dependencies = ""
for service in services:
    services_code_with_dependencies = services_code_with_dependencies + "\n" + collect_service_with_dependencies(repository, service)
#service_code_with_dependencies = collect_service_with_dependencies(repository, service, "C:/Users/ME36352/MyFolder/REPOS/Java/cfr-0.152.jar", True)

In [ ]:
 
def tree_to_yaml(root_path: str) -> str:
    """
    Build a recursive directory + file tree and return it as YAML.
    Directories are nested mappings; files are keys with null values.
    """
    root = Path(root_path)

    def build_tree(path: Path) -> dict:
        tree = {}
        # directories first, then files; both sorted alphabetically
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for item in items:
            if item.is_dir() and item.name != ".git":
                tree[item.name] = build_tree(item)
            else:
                tree[item.name] = None
        return tree

    data = {root.name: build_tree(root)}
    return yaml.safe_dump(data, sort_keys=False, default_flow_style=False)

In [ ]:
project_tree = tree_to_yaml(repository)

In [ ]:
def call_gpt(chat, dev_prompt, llm_model, reasoning):
    messages = [
       {"role": "system", "content": dev_prompt},
        {"role": "user", "content": chat}
    ]
    
    response = openai.chat.completions.create(model=llm_model, messages=messages, reasoning_effort=reasoning)
    return response.choices[0].message.content

In [ ]:
explorer_request = project_tree + "\n\n" + services_code
explorer_response = call_gpt(explorer_request, explorer_system_prompt, gpt_support, "medium")
print(explorer_response)

In [ ]:
def process_pipe_values(text):
    parts = text.split("|")
    response = "" 
    for part in parts:
        if part:  # ignores empty values from leading/trailing "|"
            response = response + collect_service_with_dependencies(repository, part)
    return response        

In [ ]:
def remove_duplicate_entities(text: str) -> str:
    """
    Removes duplicate file entities from the input text.
    Two entities are considered duplicates if they have exactly the same
    'File directory:' path line.

    Keeps the first occurrence and removes all subsequent duplicates.

    Expected input format per entity:

    File directory: some/path/file.ext

    (file content...)

    ================================================================================
    """

    separator = "=" * 80

    # Split on separator lines
    blocks = re.split(r"\n\s*={20,}\s*\n", text.strip())
    kept_blocks = []
    seen_paths = set()

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        match = re.search(r"^File directory:\s*(.+)$", block, re.MULTILINE)
        if not match:
            # If block does not match expected syntax, keep it unchanged
            kept_blocks.append(block)
            continue

        file_path = match.group(1).strip()

        if file_path in seen_paths:
            continue

        seen_paths.add(file_path)
        kept_blocks.append(block)

    return f"\n\n{separator}\n\n".join(kept_blocks)

In [ ]:
def curate_data(text: str) -> str:
    """
    Removes file entities from the input text when the file name matches
    the given exclusion criteria.

    Expected input format per entity:

    File directory: some/path/file.ext

    (file content...)

    ================================================================================
    """

    excluded_suffixes = (
        "pom.xml",
        "policy.descriptor",
        ".bat",
        ".docx",
        ".project",
        "h",
        ".yaml",
        ".json",
        ".txt",
        ".jks",
        ".log"
    )

    separator = "=" * 80

    # Split on separator lines
    blocks = re.split(r"\n\s*={20,}\s*\n", text.strip())
    kept_blocks = []

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        match = re.search(r"^File directory:\s*(.+)$", block, re.MULTILINE)
        if not match:
            # If block does not match expected syntax, keep it unchanged
            kept_blocks.append(block)
            continue

        file_path = match.group(1).strip()
        file_name = file_path.replace("\\", "/").rsplit("/", 1)[-1]

        if file_name.endswith(excluded_suffixes) or file_name in excluded_suffixes:
            continue

        kept_blocks.append(block)

    return f"\n\n{separator}\n\n".join(kept_blocks)

In [ ]:
services_code_with_dependencies = services_code_with_dependencies + "\n" + process_pipe_values(explorer_response)
services_code_with_dependencies = curate_data(services_code_with_dependencies);
services_code_with_dependencies = remove_duplicate_entities(services_code_with_dependencies)
save_string_to_txtFile(services_code_with_dependencies, "servicesAndDependencies")

main_system_prompt = main_system_prompt + services_code_with_dependencies
request = request + "\n\n" + read_directories_contents([apic_api])
save_string_to_txtFile(request, "userPrompt")

In [ ]:
gpt_response = call_gpt(request, main_system_prompt, gpt_model, "high")
save_string_to_txtFile(gpt_response, "gptResponse")

In [ ]:
def call_claude(chat):
    messages = [
        {"role": "system", "content": main_system_prompt},
        {"role": "user", "content": chat}
    ]

    response = anthropic.chat.completions.create(
        model="claude-fable-5",
        messages=messages,
        max_tokens=128000,                # REQUIRED for Anthropic
        extra_body={
            "thinking": {
                "type": "enabled",
                "budget_tokens": 126000   # must be < max_tokens
            }
        }
    )
    return response.choices[0].message.content

In [ ]:
claude_response = call_claude(request)
save_string_to_txtFile(claude_response, "claudeResponse")
print(claude_response)

In [ ]:
def call_gptCodex(prompt: str):
    response = openai.responses.create(
        model=gptCodex_model,
        reasoning={"effort": "high"},
        instructions=main_system_prompt,
        input=prompt,
    )

    return response.output_text

In [ ]:
gpt_codex_response = call_gptCodex(request)
save_string_to_txtFile(gpt_codex_response, "gptCodexResponse")
print(gpt_codex_response)

In [ ]:
def apply_llm_response(directory_path: str, llm_response: str) -> None:
    """
    Parses an LLM response of the format:

    --FN--FileName.ext--FN--
    --FC--FileContent--FC--

    ...

    ---COMMENT---
    comment text
    ---COMMENT---

    Creates all files in the specified directory and writes the comment
    into comment.txt.
    """

    target_dir = Path(directory_path)
    target_dir.mkdir(parents=True, exist_ok=True)

    # Extract comment
    comment_pattern = re.compile(
        r"---COMMENT---\s*(.*?)\s*---COMMENT---",
        re.DOTALL
    )

    comment_match = comment_pattern.search(llm_response)
    comment_text = comment_match.group(1).strip() if comment_match else ""

    # Write comment.txt
    (target_dir / "comment.txt").write_text(
        comment_text,
        encoding="utf-8"
    )

    # Remove comment section before parsing files
    content_without_comment = comment_pattern.sub("", llm_response)

    # Extract files
    file_pattern = re.compile(
        r"--FN--\s*(.*?)\s*--FN--\s*"
        r"--FC--\s*(.*?)\s*--FC--",
        re.DOTALL
    )

    for match in file_pattern.finditer(content_without_comment):
        filename = match.group(1).strip()
        file_content = match.group(2)

        file_path = target_dir / filename

        # Create parent directories if needed
        file_path.parent.mkdir(parents=True, exist_ok=True)

        file_path.write_text(file_content, encoding="utf-8")

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/postmanGenerator/outputClaude", claude_response)

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/postmanGenerator/outputGPT", gpt_response)

In [ ]:
apply_llm_response("C:/Users/ME36352/MyFolder/AI/Repos/AIRepo/postmanGenerator/outputGPTCodex", gpt_codex_response)